# Class vs Static vs Instance Methods – When and Why to Use Each

Welcome to the tenth notebook in our Object-Oriented Programming (OOP) for Computer Vision series! In this tutorial, we'll explore the different types of methods in Python classes: instance methods, class methods, and static methods. Understanding when and why to use each type of method is crucial for designing clean, maintainable, and efficient object-oriented code.

## Table of Contents
1. [Introduction to Method Types](#introduction)
2. [Instance Methods](#instance-methods)
3. [Class Methods](#class-methods)
4. [Static Methods](#static-methods)
5. [Comparison and Use Cases](#comparison)
6. [Method Types in Computer Vision](#cv-applications)
7. [Best Practices](#best-practices)
8. [Exercises](#exercises)
9. [Conclusion](#conclusion)

## 1. Introduction to Method Types <a name="introduction"></a>

Python classes can have three types of methods, each with its own purpose and behavior:

1. **Instance Methods**: The most common type, which operate on instance data and have access to the instance through the `self` parameter.
2. **Class Methods**: Methods that operate on class data and have access to the class through the `cls` parameter. They are defined using the `@classmethod` decorator.
3. **Static Methods**: Methods that don't operate on instance or class data. They are defined using the `@staticmethod` decorator and behave like regular functions that happen to be defined in a class.

Let's import the necessary libraries for our examples:

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import os
from typing import List, Dict, Tuple, Optional, Any, Union

# Helper function to display images
def display_image(image, title="Image"):
    plt.figure(figsize=(8, 6))
    if len(image.shape) == 3:  # Color image
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    else:  # Grayscale image
        plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()
    
# Create a sample image (a simple gradient with shapes)
def create_sample_image(width=300, height=200):
    # Create a gradient from black to white
    gradient = np.zeros((height, width), dtype=np.uint8)
    for i in range(width):
        gradient[:, i] = int(255 * i / width)
    
    # Convert to a color image (BGR)
    color_gradient = cv2.cvtColor(gradient, cv2.COLOR_GRAY2BGR)
    
    # Add some shapes
    cv2.circle(color_gradient, (width//4, height//2), 40, (0, 0, 255), -1)  # Red circle
    cv2.rectangle(color_gradient, (width//2, height//4), (3*width//4, 3*height//4), (0, 255, 0), -1)  # Green rectangle
    cv2.line(color_gradient, (0, 0), (width, height), (255, 0, 0), 5)  # Blue line
    
    return color_gradient

# Create a sample image
sample_image = create_sample_image()

# Display the sample image
display_image(sample_image, "Sample Image")

## 2. Instance Methods <a name="instance-methods"></a>

Instance methods are the most common type of methods in Python classes. They operate on instance data and have access to the instance through the `self` parameter.

### Key Characteristics of Instance Methods

1. They take `self` as the first parameter, which refers to the instance of the class.
2. They can access and modify instance attributes (data specific to the instance).
3. They can access class attributes (data shared among all instances).
4. They are called on instances of the class (e.g., `obj.method()`).

Let's create a simple `Image` class with instance methods:

In [ ]:
class Image:
    # Class attribute (shared among all instances)
    default_color_space = "BGR"
    
    def __init__(self, data, name="Unnamed"):
        # Instance attributes (specific to each instance)
        self.data = data
        self.name = name
        self.creation_time = datetime.now()
        
        # Compute image properties
        if data is not None:
            self.height, self.width = data.shape[:2]
            self.channels = 1 if len(data.shape) == 2 else data.shape[2]
            self.color_space = self.default_color_space if self.channels > 1 else "GRAY"
        else:
            self.height = self.width = self.channels = 0
            self.color_space = None
    
    # Instance method
    def display(self, title=None):
        """Display the image using matplotlib."""
        if self.data is None:
            print(f"Cannot display image: {self.name} (No data)")
            return self
        
        display_title = title if title else self.name
        display_image(self.data, display_title)
        return self
    
    # Instance method
    def to_grayscale(self):
        """Convert the image to grayscale."""
        if self.data is None:
            print(f"Cannot convert to grayscale: {self.name} (No data)")
            return self
        
        if self.channels == 1:
            print(f"Image is already grayscale: {self.name}")
            return self
        
        self.data = cv2.cvtColor(self.data, cv2.COLOR_BGR2GRAY)
        self.channels = 1
        self.color_space = "GRAY"
        return self
    
    # Instance method
    def apply_blur(self, kernel_size=(5, 5)):
        """Apply Gaussian blur to the image."""
        if self.data is None:
            print(f"Cannot apply blur: {self.name} (No data)")
            return self
        
        self.data = cv2.GaussianBlur(self.data, kernel_size, 0)
        return self
    
    # Instance method
    def get_info(self):
        """Get information about the image."""
        if self.data is None:
            return f"Image '{self.name}' (No data)"
        
        return f"Image '{self.name}' ({self.width}x{self.height}, {self.channels} channels, {self.color_space})"

# Create an instance of the Image class
img = Image(sample_image, "Sample")

# Call instance methods
print(img.get_info())
img.display()

# Chain instance methods
img.to_grayscale().apply_blur().display("Grayscale and Blurred")

# Access instance attributes
print(f"Image dimensions: {img.width}x{img.height}")
print(f"Creation time: {img.creation_time}")

# Access class attribute through the instance
print(f"Default color space: {img.default_color_space}")

: 

### When to Use Instance Methods

Instance methods are the most common type of methods and should be used when:

1. You need to access or modify instance attributes.
2. The method's behavior depends on the instance's state.
3. You want to enable method chaining (returning `self`).
4. The method represents an action that the instance can perform.

In the context of computer vision, instance methods are ideal for operations like:
- Image transformations (resize, rotate, filter, etc.)
- Feature extraction from a specific image
- Displaying or saving a specific image
- Computing properties of a specific image (histogram, statistics, etc.)

## 3. Class Methods <a name="class-methods"></a>

Class methods are methods that operate on class data rather than instance data. They are defined using the `@classmethod` decorator and take `cls` as the first parameter, which refers to the class itself.

### Key Characteristics of Class Methods

1. They are defined using the `@classmethod` decorator.
2. They take `cls` as the first parameter, which refers to the class itself.
3. They can access and modify class attributes (data shared among all instances).
4. They cannot access instance attributes directly (unless an instance is passed as an argument).
5. They can be called on the class itself (e.g., `Class.method()`) or on instances (e.g., `obj.method()`).

Let's extend our `Image` class with class methods:

In [ ]:
class Image:
    # Class attributes
    default_color_space = "BGR"
    supported_formats = ["jpg", "jpeg", "png", "bmp", "tiff"]
    instance_count = 0
    
    def __init__(self, data, name="Unnamed"):
        # Instance attributes
        self.data = data
        self.name = name
        self.creation_time = datetime.now()
        
        # Compute image properties
        if data is not None:
            self.height, self.width = data.shape[:2]
            self.channels = 1 if len(data.shape) == 2 else data.shape[2]
            self.color_space = self.default_color_space if self.channels > 1 else "GRAY"
        else:
            self.height = self.width = self.channels = 0
            self.color_space = None
        
        # Increment the instance count
        Image.instance_count += 1
    
    # Instance method
    def display(self, title=None):
        """Display the image using matplotlib."""
        if self.data is None:
            print(f"Cannot display image: {self.name} (No data)")
            return self
        
        display_title = title if title else self.name
        display_image(self.data, display_title)
        return self
    
    # Instance method
    def get_info(self):
        """Get information about the image."""
        if self.data is None:
            return f"Image '{self.name}' (No data)"
        
        return f"Image '{self.name}' ({self.width}x{self.height}, {self.channels} channels, {self.color_space})"
    
    # Class method
    @classmethod
    def from_file(cls, file_path):
        """Create an Image instance by loading an image from a file.
        
        Args:
            file_path (str): Path to the image file.
            
        Returns:
            Image: A new Image instance.
            
        Raises:
            ValueError: If the file format is not supported or the file cannot be read.
        """
        # Check if the file exists
        if not os.path.exists(file_path):
            raise ValueError(f"File not found: {file_path}")
        
        # Check if the file format is supported
        file_ext = os.path.splitext(file_path)[1].lower()[1:]
        if file_ext not in cls.supported_formats:
            raise ValueError(f"Unsupported file format: {file_ext}. Supported formats: {cls.supported_formats}")
        
        # Load the image
        data = cv2.imread(file_path)
        if data is None:
            raise ValueError(f"Failed to read image file: {file_path}")
        
        # Create a new instance
        return cls(data, name=os.path.basename(file_path))
    
    # Class method
    @classmethod
    def create_blank(cls, width, height, color=(0, 0, 0), name="Blank Image"):
        """Create a blank image with the specified dimensions and color.
        
        Args:
            width (int): Width of the image.
            height (int): Height of the image.
            color (tuple, optional): Color of the image (B, G, R). Defaults to (0, 0, 0) (black).
            name (str, optional): Name of the image. Defaults to "Blank Image".
            
        Returns:
            Image: A new Image instance.
        """
        # Create a blank image
        data = np.full((height, width, 3), color, dtype=np.uint8)
        
        # Create a new instance
        return cls(data, name=name)
    
    # Class method
    @classmethod
    def get_instance_count(cls):
        """Get the number of Image instances created.
        
        Returns:
            int: The number of Image instances.
        """
        return cls.instance_count
    
    # Class method
    @classmethod
    def set_default_color_space(cls, color_space):
        """Set the default color space for new images.
        
        Args:
            color_space (str): The new default color space.
        """
        cls.default_color_space = color_space
        print(f"Default color space set to: {color_space}")

# Create a sample image file if it doesn't exist
if not os.path.exists("sample_image.jpg"):
    cv2.imwrite("sample_image.jpg", sample_image)

# Use class methods
try:
    # Create an Image instance from a file
    img_from_file = Image.from_file("sample_image.jpg")
    print(img_from_file.get_info())
    img_from_file.display("Image from File")
    
    # Create a blank image
    blank_img = Image.create_blank(300, 200, color=(0, 0, 255), name="Red Image")
    print(blank_img.get_info())
    blank_img.display()
    
    # Get the instance count
    print(f"Number of Image instances: {Image.get_instance_count()}")
    
    # Set the default color space
    Image.set_default_color_space("RGB")
    print(f"New default color space: {Image.default_color_space}")
    
    # Try to load an unsupported file format
    # img_unsupported = Image.from_file("sample.txt")  # This would raise a ValueError
except ValueError as e:
    print(f"Error: {e}")

### Factory Methods

One of the most common uses of class methods is to create factory methods, which are methods that create and return instances of the class. In our example, `from_file` and `create_blank` are factory methods.

Factory methods have several advantages:

1. They provide descriptive names for different ways of creating instances.
2. They can perform validation and preprocessing before creating the instance.
3. They can return instances of subclasses based on the input.
4. They can cache or reuse instances instead of creating new ones.

Let's see how factory methods can be used to create different types of images:

In [ ]:
class Image:
    # Class attributes
    default_color_space = "BGR"
    supported_formats = ["jpg", "jpeg", "png", "bmp", "tiff"]
    instance_count = 0
    
    def __init__(self, data, name="Unnamed"):
        # Instance attributes
        self.data = data
        self.name = name
        self.creation_time = datetime.now()
        
        # Compute image properties
        if data is not None:
            self.height, self.width = data.shape[:2]
            self.channels = 1 if len(data.shape) == 2 else data.shape[2]
            self.color_space = self.default_color_space if self.channels > 1 else "GRAY"
        else:
            self.height = self.width = self.channels = 0
            self.color_space = None
        
        # Increment the instance count
        Image.instance_count += 1
    
    # Instance method
    def display(self, title=None):
        """Display the image using matplotlib."""
        if self.data is None:
            print(f"Cannot display image: {self.name} (No data)")
            return self
        
        display_title = title if title else self.name
        display_image(self.data, display_title)
        return self
    
    # Factory method for creating an image from a file
    @classmethod
    def from_file(cls, file_path):
        """Create an Image instance by loading an image from a file."""
        # Check if the file exists
        if not os.path.exists(file_path):
            raise ValueError(f"File not found: {file_path}")
        
        # Check if the file format is supported
        file_ext = os.path.splitext(file_path)[1].lower()[1:]
        if file_ext not in cls.supported_formats:
            raise ValueError(f"Unsupported file format: {file_ext}. Supported formats: {cls.supported_formats}")
        
        # Load the image
        data = cv2.imread(file_path)
        if data is None:
            raise ValueError(f"Failed to read image file: {file_path}")
        
        # Create a new instance
        return cls(data, name=os.path.basename(file_path))
    
    # Factory method for creating a blank image
    @classmethod
    def create_blank(cls, width, height, color=(0, 0, 0), name="Blank Image"):
        """Create a blank image with the specified dimensions and color."""
        # Create a blank image
        data = np.full((height, width, 3), color, dtype=np.uint8)
        
        # Create a new instance
        return cls(data, name=name)
    
    # Factory method for creating a gradient image
    @classmethod
    def create_gradient(cls, width, height, start_color=(0, 0, 0), end_color=(255, 255, 255), 
                        direction="horizontal", name="Gradient Image"):
        """Create a gradient image.
        
        Args:
            width (int): Width of the image.
            height (int): Height of the image.
            start_color (tuple, optional): Start color (B, G, R). Defaults to (0, 0, 0) (black).
            end_color (tuple, optional): End color (B, G, R). Defaults to (255, 255, 255) (white).
            direction (str, optional): Gradient direction ("horizontal" or "vertical"). Defaults to "horizontal".
            name (str, optional): Name of the image. Defaults to "Gradient Image".
            
        Returns:
            Image: A new Image instance.
            
        Raises:
            ValueError: If the direction is not "horizontal" or "vertical".
        """
        # Validate the direction
        if direction not in ["horizontal", "vertical"]:
            raise ValueError(f"Invalid direction: {direction}. Must be 'horizontal' or 'vertical'.")
        
        # Create a blank image
        data = np.zeros((height, width, 3), dtype=np.uint8)
        
        # Create the gradient
        if direction == "horizontal":
            for i in range(width):
                # Calculate the color at this position
                t = i / (width - 1)  # Normalized position (0 to 1)
                color = tuple(int(start_color[j] * (1 - t) + end_color[j] * t) for j in range(3))
                data[:, i] = color
        else:  # vertical
            for i in range(height):
                # Calculate the color at this position
                t = i / (height - 1)  # Normalized position (0 to 1)
                color = tuple(int(start_color[j] * (1 - t) + end_color[j] * t) for j in range(3))
                data[i, :] = color
        
        # Create a new instance
        return cls(data, name=name)
    
    # Factory method for creating a checkerboard image
    @classmethod
    def create_checkerboard(cls, width, height, cell_size=50, color1=(0, 0, 0), color2=(255, 255, 255), 
                           name="Checkerboard Image"):
        """Create a checkerboard image.
        
        Args:
            width (int): Width of the image.
            height (int): Height of the image.
            cell_size (int, optional): Size of each cell. Defaults to 50.
            color1 (tuple, optional): First color (B, G, R). Defaults to (0, 0, 0) (black).
            color2 (tuple, optional): Second color (B, G, R). Defaults to (255, 255, 255) (white).
            name (str, optional): Name of the image. Defaults to "Checkerboard Image".
            
        Returns:
            Image: A new Image instance.
        """
        # Create a blank image
        data = np.zeros((height, width, 3), dtype=np.uint8)
        
        # Create the checkerboard pattern
        for i in range(0, height, cell_size):
            for j in range(0, width, cell_size):
                # Determine the color of this cell
                color = color1 if ((i // cell_size) + (j // cell_size)) % 2 == 0 else color2
                
                # Fill the cell
                cell_height = min(cell_size, height - i)
                cell_width = min(cell_size, width - j)
                data[i:i+cell_height, j:j+cell_width] = color
        
        # Create a new instance
        return cls(data, name=name)

# Use the factory methods
try:
    # Create a gradient image
    gradient_img = Image.create_gradient(
        width=300, 
        height=200, 
        start_color=(0, 0, 255),  # Red
        end_color=(255, 0, 0),    # Blue
        direction="horizontal",
        name="Red to Blue Gradient"
    )
    gradient_img.display()
    
    # Create a vertical gradient image
    vertical_gradient_img = Image.create_gradient(
        width=300, 
        height=200, 
        start_color=(0, 255, 0),  # Green
        end_color=(255, 0, 255),  # Purple
        direction="vertical",
        name="Green to Purple Gradient"
    )
    vertical_gradient_img.display()
    
    # Create a checkerboard image
    checkerboard_img = Image.create_checkerboard(
        width=300,
        height=200,
        cell_size=50,
        color1=(0, 0, 0),      # Black
        color2=(0, 165, 255),  # Orange
        name="Black and Orange Checkerboard"
    )
    checkerboard_img.display()
    
    # Get the instance count
    print(f"Number of Image instances: {Image.instance_count}")
except ValueError as e:
    print(f"Error: {e}")

### When to Use Class Methods

Class methods are useful when:

1. You need to create factory methods that return instances of the class.
2. You need to access or modify class attributes (data shared among all instances).
3. You need to perform operations that are related to the class but don't depend on instance-specific data.
4. You want to create methods that can be called on the class itself, not just on instances.

In the context of computer vision, class methods are ideal for:
- Creating images from files, arrays, or other sources
- Generating synthetic images (gradients, patterns, etc.)
- Managing global settings or configurations
- Tracking statistics across all instances (e.g., counting instances)

## 4. Static Methods <a name="static-methods"></a>

Static methods are methods that don't operate on instance or class data. They are defined using the `@staticmethod` decorator and behave like regular functions that happen to be defined in a class.

### Key Characteristics of Static Methods

1. They are defined using the `@staticmethod` decorator.
2. They don't take `self` or `cls` as the first parameter.
3. They cannot access instance or class attributes directly (unless they are passed as arguments).
4. They can be called on the class itself (e.g., `Class.method()`) or on instances (e.g., `obj.method()`).
5. They are essentially regular functions that are logically grouped within a class.

Let's extend our `Image` class with static methods:

In [ ]:
class Image:
    # Class attributes
    default_color_space = "BGR"
    supported_formats = ["jpg", "jpeg", "png", "bmp", "tiff"]
    instance_count = 0
    
    def __init__(self, data, name="Unnamed"):
        # Instance attributes
        self.data = data
        self.name = name
        self.creation_time = datetime.now()
        
        # Compute image properties
        if data is not None:
            self.height, self.width = data.shape[:2]
            self.channels = 1 if len(data.shape) == 2 else data.shape[2]
            self.color_space = self.default_color_space if self.channels > 1 else "GRAY"
        else:
            self.height = self.width = self.channels = 0
            self.color_space = None
        
        # Increment the instance count
        Image.instance_count += 1
    
    # Instance method
    def display(self, title=None):
        """Display the image using matplotlib."""
        if self.data is None:
            print(f"Cannot display image: {self.name} (No data)")
            return self
        
        display_title = title if title else self.name
        display_image(self.data, display_title)
        return self
    
    # Class method
    @classmethod
    def from_file(cls, file_path):
        """Create an Image instance by loading an image from a file."""
        # Check if the file exists
        if not os.path.exists(file_path):
            raise ValueError(f"File not found: {file_path}")
        
        # Check if the file format is supported
        file_ext = os.path.splitext(file_path)[1].lower()[1:]
        if file_ext not in cls.supported_formats:
            raise ValueError(f"Unsupported file format: {file_ext}. Supported formats: {cls.supported_formats}")
        
        # Load the image
        data = cv2.imread(file_path)
        if data is None:
            raise ValueError(f"Failed to read image file: {file_path}")
        
        # Create a new instance
        return cls(data, name=os.path.basename(file_path))
    
    # Static method
    @staticmethod
    def is_valid_image_data(data):
        """Check if the data is a valid image array.
        
        Args:
            data: The data to check.
            
        Returns:
            bool: True if the data is a valid image array, False otherwise.
        """
        if not isinstance(data, np.ndarray):
            return False
        
        if len(data.shape) not in [2, 3]:
            return False
        
        if len(data.shape) == 3 and data.shape[2] not in [1, 3, 4]:
            return False
        
        return True
    
    # Static method
    @staticmethod
    def compute_psnr(img1, img2):
        """Compute the Peak Signal-to-Noise Ratio (PSNR) between two images.
        
        Args:
            img1: First image (numpy array).
            img2: Second image (numpy array).
            
        Returns:
            float: The PSNR value.
            
        Raises:
            ValueError: If the images have different shapes or are not valid image arrays.
        """
        # Check if the inputs are valid image arrays
        if not Image.is_valid_image_data(img1) or not Image.is_valid_image_data(img2):
            raise ValueError("Inputs must be valid image arrays.")
        
        # Check if the images have the same shape
        if img1.shape != img2.shape:
            raise ValueError(f"Images must have the same shape. Got {img1.shape} and {img2.shape}.")
        
        # Convert to float for calculations
        img1_float = img1.astype(np.float64)
        img2_float = img2.astype(np.float64)
        
        # Compute the Mean Squared Error (MSE)
        mse = np.mean((img1_float - img2_float) ** 2)
        if mse == 0:
            return float('inf')  # Perfect similarity
        
        # Compute the PSNR
        max_pixel = 255.0
        psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
        
        return psnr
    
    # Static method
    @staticmethod
    def compute_ssim(img1, img2):
        """Compute the Structural Similarity Index (SSIM) between two images.
        
        Args:
            img1: First image (numpy array).
            img2: Second image (numpy array).
            
        Returns:
            float: The SSIM value.
            
        Raises:
            ValueError: If the images have different shapes or are not valid image arrays.
        """
        # Check if the inputs are valid image arrays
        if not Image.is_valid_image_data(img1) or not Image.is_valid_image_data(img2):
            raise ValueError("Inputs must be valid image arrays.")
        
        # Check if the images have the same shape
        if img1.shape != img2.shape:
            raise ValueError(f"Images must have the same shape. Got {img1.shape} and {img2.shape}.")
        
        # Convert to grayscale if needed
        if len(img1.shape) == 3 and img1.shape[2] > 1:
            img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
        else:
            img1_gray = img1
        
        if len(img2.shape) == 3 and img2.shape[2] > 1:
            img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
        else:
            img2_gray = img2
        
        # Compute the SSIM
        ssim = cv2.compareSSIM(img1_gray, img2_gray)
        
        return ssim
    
    # Static method
    @staticmethod
    def blend_images(img1, img2, alpha=0.5):
        """Blend two images.
        
        Args:
            img1: First image (numpy array).
            img2: Second image (numpy array).
            alpha (float, optional): Blending factor (0.0 to 1.0). Defaults to 0.5.
            
        Returns:
            numpy.ndarray: The blended image.
            
        Raises:
            ValueError: If the images have different shapes or are not valid image arrays.
        """
        # Check if the inputs are valid image arrays
        if not Image.is_valid_image_data(img1) or not Image.is_valid_image_data(img2):
            raise ValueError("Inputs must be valid image arrays.")
        
        # Check if the images have the same shape
        if img1.shape != img2.shape:
            raise ValueError(f"Images must have the same shape. Got {img1.shape} and {img2.shape}.")
        
        # Validate alpha
        if not 0.0 <= alpha <= 1.0:
            raise ValueError(f"Alpha must be between 0.0 and 1.0. Got {alpha}.")
        
        # Blend the images
        blended = cv2.addWeighted(img1, alpha, img2, 1 - alpha, 0)
        
        return blended

# Use static methods
try:
    # Create two images
    img1 = create_sample_image(300, 200)
    img2 = create_sample_image(300, 200)  # Different random shapes
    
    # Check if the data is valid
    print(f"Is img1 valid? {Image.is_valid_image_data(img1)}")
    print(f"Is [1, 2, 3] valid? {Image.is_valid_image_data([1, 2, 3])}")
    
    # Compute the PSNR
    psnr = Image.compute_psnr(img1, img2)
    print(f"PSNR: {psnr:.2f} dB")
    
    # Blend the images
    blended = Image.blend_images(img1, img2, alpha=0.7)
    
    # Create Image instances
    img1_obj = Image(img1, "Image 1")
    img2_obj = Image(img2, "Image 2")
    blended_obj = Image(blended, "Blended Image")
    
    # Display the images
    img1_obj.display()
    img2_obj.display()
    blended_obj.display()
    
    # Try to blend images with different shapes
    # img3 = create_sample_image(400, 300)
    # blended_invalid = Image.blend_images(img1, img3)  # This would raise a ValueError
except ValueError as e:
    print(f"Error: {e}")

### When to Use Static Methods

Static methods are useful when:

1. You need a utility function that is logically related to the class but doesn't depend on instance or class state.
2. You want to group related functions within a class namespace.
3. You want to provide helper functions that operate on data of the class type but don't need access to instance or class attributes.
4. You want to create pure functions (functions without side effects) that are related to the class.

In the context of computer vision, static methods are ideal for:
- Image processing utilities (blending, comparing, etc.)
- Validation functions
- Mathematical operations on image data
- Conversion functions between different formats or representations

## 5. Comparison and Use Cases <a name="comparison"></a>

Let's compare the three types of methods and their use cases:

| Feature | Instance Method | Class Method | Static Method |
|---------|----------------|--------------|---------------|
| Decorator | None | `@classmethod` | `@staticmethod` |
| First Parameter | `self` (instance) | `cls` (class) | None |
| Can Access Instance Attributes | Yes | No (unless instance is passed) | No (unless instance is passed) |
| Can Access Class Attributes | Yes | Yes | No (unless class is passed) |
| Can Modify Instance State | Yes | No (unless instance is passed) | No (unless instance is passed) |
| Can Modify Class State | Yes | Yes | No (unless class is passed) |
| Called On | Instance | Class or Instance | Class or Instance |
| Typical Use Cases | Operations on instance data, transformations, actions | Factory methods, class-level operations, tracking | Utility functions, pure functions, helpers |

### Decision Tree for Choosing Method Type

Here's a simple decision tree to help you choose the right type of method:

1. Does the method need to access or modify instance attributes?
   - Yes: Use an **instance method**.
   - No: Continue to step 2.

2. Does the method need to access or modify class attributes?
   - Yes: Use a **class method**.
   - No: Continue to step 3.

3. Is the method logically related to the class but doesn't depend on instance or class state?
   - Yes: Use a **static method**.
   - No: Consider whether the function should be defined outside the class.

## 6. Method Types in Computer Vision <a name="cv-applications"></a>

Let's see how the different types of methods can be used in a computer vision context by creating a comprehensive `ImageProcessor` class:

In [ ]:
class ImageProcessor:
    # Class attributes
    default_kernel_size = (5, 5)
    default_sigma = 1.0
    instance_count = 0
    
    def __init__(self, name="Unnamed Processor"):
        # Instance attributes
        self.name = name
        self.creation_time = datetime.now()
        self.processing_history = []
        
        # Increment the instance count
        ImageProcessor.instance_count += 1
    
    # Instance method
    def process(self, image, operations=None):
        """Process an image with a sequence of operations.
        
        Args:
            image: The image to process (numpy array).
            operations (list, optional): List of operations to apply. Each operation is a tuple
                of (operation_name, parameters_dict). Defaults to None.
                
        Returns:
            numpy.ndarray: The processed image.
        """
        # Validate the input
        if not ImageProcessor.is_valid_image(image):
            raise ValueError("Input must be a valid image array.")
        
        # Make a copy of the input image
        result = image.copy()
        
        # If no operations are specified, use default operations
        if operations is None:
            operations = [
                ("grayscale", {}),
                ("blur", {"kernel_size": self.default_kernel_size, "sigma": self.default_sigma})
            ]
        
        # Apply each operation
        for op_name, op_params in operations:
            # Apply the operation
            if op_name == "grayscale":
                if len(result.shape) == 3 and result.shape[2] > 1:
                    result = cv2.cvtColor(result, cv2.COLOR_BGR2GRAY)
            elif op_name == "blur":
                kernel_size = op_params.get("kernel_size", self.default_kernel_size)
                sigma = op_params.get("sigma", self.default_sigma)
                result = cv2.GaussianBlur(result, kernel_size, sigma)
            elif op_name == "threshold":
                thresh = op_params.get("thresh", 127)
                maxval = op_params.get("maxval", 255)
                type = op_params.get("type", cv2.THRESH_BINARY)
                
                # Convert to grayscale if needed
                if len(result.shape) == 3 and result.shape[2] > 1:
                    gray = cv2.cvtColor(result, cv2.COLOR_BGR2GRAY)
                else:
                    gray = result
                
                _, result = cv2.threshold(gray, thresh, maxval, type)
            elif op_name == "canny":
                threshold1 = op_params.get("threshold1", 100)
                threshold2 = op_params.get("threshold2", 200)
                
                # Convert to grayscale if needed
                if len(result.shape) == 3 and result.shape[2] > 1:
                    gray = cv2.cvtColor(result, cv2.COLOR_BGR2GRAY)
                else:
                    gray = result
                
                result = cv2.Canny(gray, threshold1, threshold2)
            else:
                raise ValueError(f"Unknown operation: {op_name}")
            
            # Log the operation
            self.processing_history.append({
                "operation": op_name,
                "parameters": op_params,
                "timestamp": datetime.now()
            })
        
        return result
    
    # Instance method
    def get_history(self):
        """Get the processing history.
        
        Returns:
            list: The processing history.
        """
        return self.processing_history
    
    # Instance method
    def clear_history(self):
        """Clear the processing history.
        
        Returns:
            ImageProcessor: self for method chaining.
        """
        self.processing_history = []
        return self
    
    # Class method
    @classmethod
    def get_instance_count(cls):
        """Get the number of ImageProcessor instances created.
        
        Returns:
            int: The number of ImageProcessor instances.
        """
        return cls.instance_count
    
    # Class method
    @classmethod
    def set_default_kernel_size(cls, kernel_size):
        """Set the default kernel size for blur operations.
        
        Args:
            kernel_size (tuple): The new default kernel size.
        """
        cls.default_kernel_size = kernel_size
        print(f"Default kernel size set to: {kernel_size}")
    
    # Class method
    @classmethod
    def create_edge_detector(cls, threshold1=100, threshold2=200):
        """Create an ImageProcessor configured for edge detection.
        
        Args:
            threshold1 (int, optional): First threshold for the Canny edge detector. Defaults to 100.
            threshold2 (int, optional): Second threshold for the Canny edge detector. Defaults to 200.
            
        Returns:
            ImageProcessor: A new ImageProcessor instance.
        """
        processor = cls("Edge Detector")
        
        # Define the edge detection operations
        processor.default_operations = [
            ("grayscale", {}),
            ("blur", {"kernel_size": cls.default_kernel_size, "sigma": cls.default_sigma}),
            ("canny", {"threshold1": threshold1, "threshold2": threshold2})
        ]
        
        return processor
    
    # Class method
    @classmethod
    def create_thresholder(cls, thresh=127, maxval=255, type=cv2.THRESH_BINARY):
        """Create an ImageProcessor configured for thresholding.
        
        Args:
            thresh (int, optional): Threshold value. Defaults to 127.
            maxval (int, optional): Maximum value. Defaults to 255.
            type (int, optional): Thresholding type. Defaults to cv2.THRESH_BINARY.
            
        Returns:
            ImageProcessor: A new ImageProcessor instance.
        """
        processor = cls("Thresholder")
        
        # Define the thresholding operations
        processor.default_operations = [
            ("grayscale", {}),
            ("threshold", {"thresh": thresh, "maxval": maxval, "type": type})
        ]
        
        return processor
    
    # Static method
    @staticmethod
    def is_valid_image(image):
        """Check if the data is a valid image array.
        
        Args:
            image: The data to check.
            
        Returns:
            bool: True if the data is a valid image array, False otherwise.
        """
        if not isinstance(image, np.ndarray):
            return False
        
        if len(image.shape) not in [2, 3]:
            return False
        
        if len(image.shape) == 3 and image.shape[2] not in [1, 3, 4]:
            return False
        
        return True
    
    # Static method
    @staticmethod
    def compare_images(img1, img2, method="psnr"):
        """Compare two images using the specified method.
        
        Args:
            img1: First image (numpy array).
            img2: Second image (numpy array).
            method (str, optional): Comparison method ("psnr" or "ssim"). Defaults to "psnr".
            
        Returns:
            float: The comparison result.
            
        Raises:
            ValueError: If the images have different shapes or are not valid image arrays.
        """
        # Check if the inputs are valid image arrays
        if not ImageProcessor.is_valid_image(img1) or not ImageProcessor.is_valid_image(img2):
            raise ValueError("Inputs must be valid image arrays.")
        
        # Check if the images have the same shape
        if img1.shape != img2.shape:
            raise ValueError(f"Images must have the same shape. Got {img1.shape} and {img2.shape}.")
        
        if method.lower() == "psnr":
            # Convert to float for calculations
            img1_float = img1.astype(np.float64)
            img2_float = img2.astype(np.float64)
            
            # Compute the Mean Squared Error (MSE)
            mse = np.mean((img1_float - img2_float) ** 2)
            if mse == 0:
                return float('inf')  # Perfect similarity
            
            # Compute the PSNR
            max_pixel = 255.0
            psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
            
            return psnr
        
        elif method.lower() == "ssim":
            # Convert to grayscale if needed
            if len(img1.shape) == 3 and img1.shape[2] > 1:
                img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
            else:
                img1_gray = img1
            
            if len(img2.shape) == 3 and img2.shape[2] > 1:
                img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
            else:
                img2_gray = img2
            
            # Compute the SSIM
            ssim = cv2.compareSSIM(img1_gray, img2_gray)
            
            return ssim
        
        else:
            raise ValueError(f"Unknown comparison method: {method}. Supported methods: 'psnr', 'ssim'.")
    
    # Static method
    @staticmethod
    def display_image(image, title="Image"):
        """Display an image using matplotlib.
        
        Args:
            image: The image to display (numpy array).
            title (str, optional): Title for the image. Defaults to "Image".
        """
        if not ImageProcessor.is_valid_image(image):
            raise ValueError("Input must be a valid image array.")
        
        display_image(image, title)

# Use the ImageProcessor class
try:
    # Create an instance
    processor = ImageProcessor("Basic Processor")
    
    # Process an image using instance methods
    result = processor.process(sample_image, [
        ("grayscale", {}),
        ("blur", {"kernel_size": (15, 15), "sigma": 2.0})
    ])
    
    # Display the result using a static method
    ImageProcessor.display_image(result, "Processed Image")
    
    # Print the processing history
    print("Processing History:")
    for i, entry in enumerate(processor.get_history()):
        print(f"{i+1}. {entry['operation']} - {entry['parameters']}")
    
    # Create specialized processors using class methods
    edge_detector = ImageProcessor.create_edge_detector(threshold1=50, threshold2=150)
    thresholder = ImageProcessor.create_thresholder(thresh=100, type=cv2.THRESH_BINARY_INV)
    
    # Process the image with the specialized processors
    edges = edge_detector.process(sample_image, [
        ("grayscale", {}),
        ("blur", {"kernel_size": (5, 5), "sigma": 1.5}),
        ("canny", {"threshold1": 50, "threshold2": 150})
    ])
    
    threshold = thresholder.process(sample_image, [
        ("grayscale", {}),
        ("threshold", {"thresh": 100, "maxval": 255, "type": cv2.THRESH_BINARY_INV})
    ])
    
    # Display the results
    ImageProcessor.display_image(edges, "Edge Detection")
    ImageProcessor.display_image(threshold, "Thresholding")
    
    # Use a class method to set the default kernel size
    ImageProcessor.set_default_kernel_size((7, 7))
    
    # Get the instance count using a class method
    print(f"Number of ImageProcessor instances: {ImageProcessor.get_instance_count()}")
    
    # Compare images using a static method
    psnr = ImageProcessor.compare_images(sample_image, result, method="psnr")
    print(f"PSNR between original and processed: {psnr:.2f} dB")
except ValueError as e:
    print(f"Error: {e}")

## 7. Best Practices <a name="best-practices"></a>

Here are some best practices for using the different types of methods in Python classes:

### General Best Practices

1. **Choose the Right Method Type**: Use the appropriate method type based on whether the method needs to access instance data, class data, or neither.
2. **Consistent Naming**: Use consistent naming conventions for different types of methods.
3. **Clear Documentation**: Document the purpose and behavior of each method, especially for class and static methods.
4. **Single Responsibility**: Each method should have a single responsibility and do one thing well.

### Instance Methods

1. **Return `self` for Method Chaining**: Return `self` from methods that modify the instance to enable method chaining.
2. **Validate Inputs**: Validate inputs to ensure they meet the method's requirements.
3. **Avoid Side Effects**: Minimize side effects on other objects or global state.
4. **Use Private Methods**: Use private methods (prefixed with an underscore) for internal implementation details.

### Class Methods

1. **Use for Factory Methods**: Use class methods for factory methods that create and return instances of the class.
2. **Use for Class-Level Operations**: Use class methods for operations that affect all instances of the class.
3. **Use `cls` Parameter**: Use the `cls` parameter to create instances of the class or access class attributes.
4. **Avoid Instance-Specific Logic**: Avoid logic that depends on instance-specific data.

### Static Methods

1. **Use for Utility Functions**: Use static methods for utility functions that are logically related to the class.
2. **Keep Pure**: Keep static methods pure (no side effects) whenever possible.
3. **Consider Module-Level Functions**: If a static method doesn't need to be in the class, consider making it a module-level function.
4. **Avoid Accessing Class or Instance State**: Avoid accessing class or instance state directly; pass any needed data as arguments.

## 8. Exercises <a name="exercises"></a>

Now that you've learned about the different types of methods in Python classes, try these exercises to reinforce your understanding:

### Exercise 1: Create a Feature Detector Class

Create a `FeatureDetector` class with the following methods:

1. **Instance Methods**:
   - `detect_features(self, image)`: Detect features in the image using a specified algorithm.
   - `draw_features(self, image, keypoints)`: Draw the detected features on the image.
   - `get_feature_count(self)`: Get the number of features detected in the last call to `detect_features`.

2. **Class Methods**:
   - `create_sift_detector(cls, nfeatures=0, n_octave_layers=3)`: Create a detector using the SIFT algorithm.
   - `create_orb_detector(cls, nfeatures=500, scale_factor=1.2)`: Create a detector using the ORB algorithm.
   - `get_available_algorithms(cls)`: Get a list of available feature detection algorithms.

3. **Static Methods**:
   - `match_features(keypoints1, descriptors1, keypoints2, descriptors2)`: Match features between two images.
   - `filter_matches(matches, ratio=0.75)`: Filter matches based on the ratio test.
   - `draw_matches(img1, keypoints1, img2, keypoints2, matches)`: Draw the matches between two images.

### Exercise 2: Create an Image Transformation Class

Create an `ImageTransformer` class with the following methods:

1. **Instance Methods**:
   - `rotate(self, image, angle)`: Rotate the image by the specified angle.
   - `resize(self, image, width, height)`: Resize the image to the specified dimensions.
   - `crop(self, image, x, y, width, height)`: Crop a region from the image.
   - `get_transformation_history(self)`: Get the history of transformations applied.

2. **Class Methods**:
   - `create_with_defaults(cls, rotation_interpolation=cv2.INTER_LINEAR)`: Create a transformer with default settings.
   - `set_default_interpolation(cls, interpolation)`: Set the default interpolation method for all instances.
   - `get_instance_count(cls)`: Get the number of transformer instances created.

3. **Static Methods**:
   - `compute_rotation_matrix(angle, center=None)`: Compute a rotation matrix for the specified angle.
   - `compute_affine_transform(src_points, dst_points)`: Compute an affine transformation matrix.
   - `apply_transformation(image, matrix)`: Apply a transformation matrix to an image.

### Exercise 3: Create an Image Filter Class

Create an `ImageFilter` class with the following methods:

1. **Instance Methods**:
   - `apply(self, image)`: Apply the filter to the image.
   - `set_parameters(self, **kwargs)`: Set the filter parameters.
   - `get_parameters(self)`: Get the current filter parameters.
   - `reset_parameters(self)`: Reset the parameters to their default values.

2. **Class Methods**:
   - `create_gaussian_filter(cls, kernel_size=(5, 5), sigma=1.0)`: Create a Gaussian filter.
   - `create_median_filter(cls, kernel_size=5)`: Create a median filter.
   - `create_bilateral_filter(cls, d=9, sigma_color=75, sigma_space=75)`: Create a bilateral filter.
   - `get_available_filters(cls)`: Get a list of available filter types.

3. **Static Methods**:
   - `create_kernel(kernel_type, kernel_size)`: Create a kernel of the specified type and size.
   - `apply_kernel(image, kernel)`: Apply a kernel to an image using convolution.
   - `normalize_kernel(kernel)`: Normalize a kernel so that its elements sum to 1.

## 9. Conclusion <a name="conclusion"></a>

In this notebook, we've explored the three types of methods in Python classes: instance methods, class methods, and static methods. We've seen how each type of method serves a different purpose and when to use each one.

Key takeaways:

1. **Instance Methods** operate on instance data and have access to the instance through the `self` parameter. They are the most common type of method and are used for operations that depend on instance state.

2. **Class Methods** operate on class data and have access to the class through the `cls` parameter. They are defined using the `@classmethod` decorator and are commonly used for factory methods and class-level operations.

3. **Static Methods** don't operate on instance or class data. They are defined using the `@staticmethod` decorator and are essentially regular functions that are logically grouped within a class.

4. The choice of method type depends on whether the method needs to access instance data, class data, or neither.

5. In the context of computer vision, each type of method has its own use cases, such as image transformations (instance methods), factory methods for creating specialized processors (class methods), and utility functions for comparing or blending images (static methods).

Understanding the differences between these method types and when to use each one is crucial for designing clean, maintainable, and efficient object-oriented code in Python.